# 03. 텍스트 유사도 (Text Similarity) 🏪

## 학습 목표
- 다양한 텍스트 유사도 측정 방법을 이해하고 직접 구현
- ai-ipsonum에서 사용하는 Bigram Dice coefficient 방식 분석
- 한국어 매장명 매칭에 가장 적합한 방식 탐색

## ai-ipsonum 연계 🏪
- 현재 ai-ipsonum은 **Bigram Dice coefficient**로 매장명 매칭
- 참고: `ai-ipsonum/src/lib/parser/fuzzy-match.ts`

---

In [ ]:
import numpy as np
from collections import Counter
import re
import matplotlib.pyplot as plt

## 1. 문자열 유사도: Edit Distance (Levenshtein)

두 문자열을 **같게 만들기 위해 필요한 최소 편집 횟수**.

편집 연산:
- **삽입** (Insert): "" → "a"
- **삭제** (Delete): "a" → ""
- **교체** (Replace): "a" → "b"

### DP 점화식

문자열 $s_1$ (길이 $m$)과 $s_2$ (길이 $n$)에 대해:

$$D[i][j] = \begin{cases} j & \text{if } i = 0 \\ i & \text{if } j = 0 \\ D[i-1][j-1] & \text{if } s_1[i] = s_2[j] \\ 1 + \min(D[i-1][j], D[i][j-1], D[i-1][j-1]) & \text{otherwise} \end{cases}$$

- $D[i-1][j] + 1$: 삭제
- $D[i][j-1] + 1$: 삽입
- $D[i-1][j-1] + 1$: 교체

In [ ]:
def levenshtein_distance(s1, s2):
    """Edit Distance를 DP로 구현"""
    m, n = len(s1), len(s2)
    
    # DP 테이블 초기화
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    # 베이스 케이스
    for i in range(m + 1):
        dp[i][0] = i  # s1의 i글자를 모두 삭제
    for j in range(n + 1):
        dp[0][j] = j  # s2의 j글자를 모두 삽입
    
    # DP 채우기
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]  # 같으면 비용 0
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # 삭제
                    dp[i][j-1],    # 삽입
                    dp[i-1][j-1]   # 교체
                )
    
    return dp[m][n], dp


# 테스트
s1, s2 = "kitten", "sitting"
dist, dp = levenshtein_distance(s1, s2)
print(f"Edit Distance('{s1}', '{s2}') = {dist}")
print(f"  kitten → sitten (k→s 교체)")
print(f"  sitten → sittin (e→i 교체)")
print(f"  sittin → sitting (g 삽입)")
print(f"  → 총 3번의 편집")

In [ ]:
# DP 테이블 시각화
def visualize_dp_table(s1, s2, dp):
    fig, ax = plt.subplots(figsize=(8, 6))
    dp_arr = np.array(dp)
    im = ax.imshow(dp_arr, cmap='YlOrRd')
    
    # 축 레이블
    ax.set_xticks(range(len(s2) + 1))
    ax.set_xticklabels([''] + list(s2))
    ax.set_yticks(range(len(s1) + 1))
    ax.set_yticklabels([''] + list(s1))
    
    # 셀에 값 표시
    for i in range(len(s1) + 1):
        for j in range(len(s2) + 1):
            ax.text(j, i, str(dp_arr[i, j]), ha='center', va='center', fontsize=12)
    
    ax.set_xlabel('s2: ' + s2)
    ax.set_ylabel('s1: ' + s1)
    ax.set_title(f'Edit Distance DP Table\nDistance = {dp_arr[-1, -1]}')
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()

visualize_dp_table(s1, s2, dp)

In [ ]:
# 정규화된 Edit Distance (0~1 사이의 유사도로 변환)
def edit_similarity(s1, s2):
    dist, _ = levenshtein_distance(s1, s2)
    max_len = max(len(s1), len(s2))
    if max_len == 0:
        return 1.0
    return 1 - dist / max_len

# 테스트
test_pairs = [
    ("hello", "hello"),
    ("hello", "hallo"),
    ("hello", "world"),
    ("cat", "cats"),
]

print("Edit Similarity (1에 가까울수록 유사):")
for s1, s2 in test_pairs:
    dist, _ = levenshtein_distance(s1, s2)
    sim = edit_similarity(s1, s2)
    print(f"  '{s1}' vs '{s2}': distance={dist}, similarity={sim:.4f}")

---
## 2. N-gram 기반 유사도

N-gram: 연속된 N개의 문자(또는 단어)를 하나의 단위로 취급.

| N | 이름 | "hello"의 n-gram |
|---|------|-------------------|
| 1 | Unigram | {h, e, l, l, o} |
| 2 | Bigram | {he, el, ll, lo} |
| 3 | Trigram | {hel, ell, llo} |

### 2.1 Jaccard Similarity

두 집합의 교집합 / 합집합:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

### 2.2 Dice Coefficient

$$D(A, B) = \frac{2|A \cap B|}{|A| + |B|}$$

Jaccard와 비슷하지만, 분모가 다르다. Dice가 보통 더 높은 값을 준다.

In [ ]:
def get_ngrams(text, n=2):
    """문자 단위 n-gram 생성"""
    return [text[i:i+n] for i in range(len(text) - n + 1)]

def jaccard_similarity(s1, s2, n=2):
    """Jaccard Similarity (n-gram 기반)"""
    ngrams1 = set(get_ngrams(s1, n))
    ngrams2 = set(get_ngrams(s2, n))
    
    if not ngrams1 and not ngrams2:
        return 1.0
    
    intersection = ngrams1 & ngrams2
    union = ngrams1 | ngrams2
    
    return len(intersection) / len(union)

def dice_coefficient(s1, s2, n=2):
    """Dice Coefficient (n-gram 기반)"""
    ngrams1 = set(get_ngrams(s1, n))
    ngrams2 = set(get_ngrams(s2, n))
    
    if not ngrams1 and not ngrams2:
        return 1.0
    if not ngrams1 or not ngrams2:
        return 0.0
    
    intersection = ngrams1 & ngrams2
    
    return 2 * len(intersection) / (len(ngrams1) + len(ngrams2))


# 테스트
s1 = "night"
s2 = "nacht"

bigrams1 = get_ngrams(s1, 2)
bigrams2 = get_ngrams(s2, 2)

print(f"'{s1}' bigrams: {bigrams1}")
print(f"'{s2}' bigrams: {bigrams2}")
print(f"교집합: {set(bigrams1) & set(bigrams2)}")
print(f"합집합: {set(bigrams1) | set(bigrams2)}")
print(f"\nJaccard: {jaccard_similarity(s1, s2):.4f}")
print(f"Dice:    {dice_coefficient(s1, s2):.4f}")

---
## 3. Bigram Dice Coefficient 🏪

**ai-ipsonum에서 사용하는 방식**: 매장명 매칭에 Bigram Dice를 사용.

### 왜 Bigram Dice인가?

- **띄어쓰기에 강함**: bigram 단위로 쪼개므로 공백 변형에 덜 민감
- **부분 매칭 가능**: 전체가 아닌 부분적 일치도 점수에 반영
- **구현이 단순**: 복잡한 모델 없이도 합리적인 성능
- **빠른 속도**: O(n) 시간 복잡도

### ai-ipsonum 스타일 구현

In [ ]:
def bigram_dice(s1, s2):
    """
    ai-ipsonum 스타일 Bigram Dice Coefficient
    - 공백 제거 후 bigram 생성
    - 대소문자 무시
    """
    # 정규화: 소문자화 + 공백 제거
    s1 = s1.lower().replace(' ', '')
    s2 = s2.lower().replace(' ', '')
    
    # 길이 1 이하인 경우
    if len(s1) <= 1 or len(s2) <= 1:
        return 1.0 if s1 == s2 else 0.0
    
    # Bigram 생성
    bigrams1 = [s1[i:i+2] for i in range(len(s1) - 1)]
    bigrams2 = [s2[i:i+2] for i in range(len(s2) - 1)]
    
    # 멀티셋 교집합 (같은 bigram이 여러 번 나올 수 있으므로)
    counter1 = Counter(bigrams1)
    counter2 = Counter(bigrams2)
    
    intersection = sum((counter1 & counter2).values())
    total = len(bigrams1) + len(bigrams2)
    
    return 2 * intersection / total


# ai-ipsonum에서 실제로 마주칠 매장명 매칭 케이스
print("=== Bigram Dice: 매장명 매칭 예시 ===")
test_cases = [
    ("스타벅스", "스타벅스"),
    ("스타벅스 강남점", "스타벅스"),
    ("스타벅스", "스타 벅스"),
    ("블루보틀", "블루 보틀"),
    ("맥도날드", "맥도널드"),
    ("이디야커피", "이디야"),
    ("GS25 강남점", "GS25"),
    ("CU 편의점", "CU"),
]

for s1, s2 in test_cases:
    score = bigram_dice(s1, s2)
    print(f"  '{s1}' vs '{s2}': {score:.4f}")

In [ ]:
# Bigram Dice 과정을 상세하게 보여주기
def bigram_dice_verbose(s1, s2):
    """상세 과정 출력"""
    s1_norm = s1.lower().replace(' ', '')
    s2_norm = s2.lower().replace(' ', '')
    
    bg1 = [s1_norm[i:i+2] for i in range(len(s1_norm) - 1)]
    bg2 = [s2_norm[i:i+2] for i in range(len(s2_norm) - 1)]
    
    c1 = Counter(bg1)
    c2 = Counter(bg2)
    intersection = c1 & c2
    
    print(f"  s1 정규화: '{s1}' → '{s1_norm}'")
    print(f"  s2 정규화: '{s2}' → '{s2_norm}'")
    print(f"  s1 bigrams: {bg1}")
    print(f"  s2 bigrams: {bg2}")
    print(f"  교집합: {dict(intersection)}")
    print(f"  Dice = 2*{sum(intersection.values())} / ({len(bg1)}+{len(bg2)}) = {2*sum(intersection.values()) / (len(bg1)+len(bg2)):.4f}")

print("\n--- '스타벅스 강남점' vs '스타벅스' ---")
bigram_dice_verbose("스타벅스 강남점", "스타벅스")

print("\n--- '블루보틀' vs '블루 보틀' ---")
bigram_dice_verbose("블루보틀", "블루 보틀")

---
## 4. Cosine Similarity

### 4.1 TF-IDF 기반

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_cosine_similarity(s1, s2):
    """TF-IDF 벡터의 코사인 유사도"""
    # 한국어 문자 단위 분석 (analyzer='char')
    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 2))
    tfidf = vectorizer.fit_transform([s1, s2]).toarray()
    
    cos_sim = np.dot(tfidf[0], tfidf[1]) / (
        np.linalg.norm(tfidf[0]) * np.linalg.norm(tfidf[1])
    )
    return cos_sim

print("=== TF-IDF Cosine Similarity ===")
for s1, s2 in test_cases:
    score = tfidf_cosine_similarity(s1, s2)
    print(f"  '{s1}' vs '{s2}': {score:.4f}")

### 4.2 Embedding 기반

문장/단어를 Dense Embedding으로 변환한 뒤 코사인 유사도를 계산.
실제로는 Sentence-BERT 등의 모델을 사용하지만, 여기서는 간단한 문자 임베딩으로 시연.

In [ ]:
def char_embedding_similarity(s1, s2, dim=50):
    """
    간단한 문자 임베딩 기반 유사도.
    각 문자를 해시 기반으로 고정 차원 벡터에 매핑한 뒤 평균.
    (실전에서는 Sentence-BERT 등의 모델을 사용)
    """
    def text_to_vec(text, dim):
        np.random.seed(0)  # 재현성
        vec = np.zeros(dim)
        for ch in text:
            np.random.seed(ord(ch))  # 같은 문자 → 같은 벡터
            vec += np.random.randn(dim)
        if len(text) > 0:
            vec /= len(text)
        return vec
    
    v1 = text_to_vec(s1.replace(' ', ''), dim)
    v2 = text_to_vec(s2.replace(' ', ''), dim)
    
    cos_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return cos_sim

print("=== Char Embedding Cosine Similarity ===")
for s1, s2 in test_cases:
    score = char_embedding_similarity(s1, s2)
    print(f"  '{s1}' vs '{s2}': {score:.4f}")

---
## 5. 한국어 매장명 매칭 실험 🏪

ai-ipsonum 실전 시나리오: 카드 명세서의 매장명을 표준 매장명과 매칭.

### 도전 과제
- 띄어쓰기 변형: "스타벅스" vs "스타 벅스"
- 지점명 포함: "스타벅스 강남점" vs "스타벅스"
- 오타/변형: "맥도날드" vs "맥도널드"
- 영문 혼용: "GS25" vs "지에스25"
- 부분 매칭: "이디야커피" vs "이디야"

In [ ]:
# 전체 테스트 데이터셋
match_tests = [
    # (쿼리, 정답, 카테고리)
    ("스타벅스 강남점", "스타벅스", "지점명 제거"),
    ("스타벅스강남점", "스타벅스", "지점명 제거"),
    ("스타 벅스", "스타벅스", "띄어쓰기"),
    ("블루 보틀", "블루보틀", "띄어쓰기"),
    ("맥도날드", "맥도널드", "표기 변형"),
    ("이디야커피", "이디야", "부분 매칭"),
    ("투썸플레이스 역삼점", "투썸플레이스", "지점명 제거"),
    ("CU편의점", "CU", "부분 매칭"),
    ("GS25 강남역점", "GS25", "지점명 제거"),
    ("서브웨이 삼성역", "서브웨이", "지점명 제거"),
]

# 후보 매장명 리스트
store_names = [
    "스타벅스", "블루보틀", "맥도널드", "이디야", "투썸플레이스",
    "CU", "GS25", "서브웨이", "배스킨라빈스", "던킨도너츠",
    "빽다방", "파리바게뜨", "롯데리아", "버거킹", "KFC"
]

In [ ]:
# 정규화 전처리 함수
def normalize_store_name(name):
    """매장명 정규화"""
    # 지점명 패턴 제거
    name = re.sub(r'\s*(강남|역삼|삼성|서초|잠실|홍대|신촌|건대|합정|성수)[가-힣]*점?\s*$', '', name)
    name = re.sub(r'\s*[가-힣]*[점역]\s*$', '', name)
    # 공백 제거
    name = name.replace(' ', '')
    # 소문자화
    name = name.lower()
    return name

# 정규화 효과 확인
print("=== 정규화 전처리 ===")
for query, answer, cat in match_tests[:5]:
    print(f"  '{query}' → '{normalize_store_name(query)}'")

In [ ]:
# 모든 유사도 방식 비교 실험
def run_matching_experiment(match_tests, store_names):
    """각 유사도 방식으로 매칭 실험"""
    methods = {
        'Edit Distance': lambda s1, s2: edit_similarity(s1.replace(' ', ''), s2.replace(' ', '')),
        'Jaccard (bigram)': lambda s1, s2: jaccard_similarity(s1.replace(' ', ''), s2.replace(' ', ''), n=2),
        'Dice (bigram)': lambda s1, s2: bigram_dice(s1, s2),
        'TF-IDF Cosine': lambda s1, s2: tfidf_cosine_similarity(s1, s2),
    }
    
    results = {method: {'correct': 0, 'total': 0, 'details': []} for method in methods}
    
    for query, answer, category in match_tests:
        for method_name, method_fn in methods.items():
            # 모든 후보와 유사도 계산
            scores = []
            for store in store_names:
                score = method_fn(query, store)
                scores.append((store, score))
            
            # 가장 높은 점수의 매장
            best_match, best_score = max(scores, key=lambda x: x[1])
            is_correct = best_match == answer
            
            results[method_name]['total'] += 1
            if is_correct:
                results[method_name]['correct'] += 1
            results[method_name]['details'].append({
                'query': query,
                'answer': answer,
                'predicted': best_match,
                'score': best_score,
                'correct': is_correct,
                'category': category
            })
    
    return results

results = run_matching_experiment(match_tests, store_names)

In [ ]:
# 결과 출력
print("=== 매장명 매칭 실험 결과 ===")
print()

# 정확도 비교 테이블
print(f"{'방식':<20} {'정확도':>8}")
print("-" * 30)
for method, data in results.items():
    acc = data['correct'] / data['total'] * 100
    print(f"{method:<20} {acc:>7.1f}%")

# 상세 결과
print(f"\n\n=== 상세 결과 (Bigram Dice) ===")
print(f"{'쿼리':<20} {'정답':<12} {'예측':<12} {'점수':>6} {'결과'}")
print("-" * 60)
for d in results['Dice (bigram)']['details']:
    status = 'O' if d['correct'] else 'X'
    print(f"{d['query']:<20} {d['answer']:<12} {d['predicted']:<12} {d['score']:>6.3f} {status}")

In [ ]:
# 방식별 정확도 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 전체 정확도 비교
ax = axes[0]
methods = list(results.keys())
accuracies = [results[m]['correct'] / results[m]['total'] * 100 for m in methods]
bars = ax.bar(range(len(methods)), accuracies, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Matching Accuracy by Method')
ax.set_ylim(0, 110)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
           f'{acc:.0f}%', ha='center', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# 오른쪽: 카테고리별 분석 (Bigram Dice)
ax = axes[1]
dice_details = results['Dice (bigram)']['details']
categories = {}
for d in dice_details:
    cat = d['category']
    if cat not in categories:
        categories[cat] = {'correct': 0, 'total': 0}
    categories[cat]['total'] += 1
    if d['correct']:
        categories[cat]['correct'] += 1

cat_names = list(categories.keys())
cat_accs = [categories[c]['correct'] / categories[c]['total'] * 100 for c in cat_names]
bars = ax.barh(range(len(cat_names)), cat_accs, color='#C44E52')
ax.set_yticks(range(len(cat_names)))
ax.set_yticklabels(cat_names)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Bigram Dice: Accuracy by Category')
ax.set_xlim(0, 110)
for bar, acc in zip(bars, cat_accs):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
           f'{acc:.0f}%', va='center', fontsize=11)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

---
## 6. 분석: 한국어 매장명 매칭에 가장 적합한 방식

### 방식별 장단점

| 방식 | 띄어쓰기 | 지점명 | 오타 | 속도 | 종합 |
|------|----------|--------|------|------|------|
| Edit Distance | 약함 (공백 제거 시 개선) | 약함 | 강함 | 느림 O(mn) | 오타에 강하지만 부분 매칭 약함 |
| Jaccard Bigram | 보통 | 보통 | 보통 | 빠름 | 균형잡힌 성능 |
| **Dice Bigram** | **강함** | **보통** | **보통** | **빠름** | **실용적 선택 (ai-ipsonum)** |
| TF-IDF Cosine | 약함 | 강함 | 약함 | 느림 | 긴 텍스트에 적합 |

### 개선 방향 (ai-ipsonum)

1. **전처리 결합**: 지점명 패턴 제거 + Bigram Dice
2. **임계값 설정**: Dice > 0.6 이상이면 매칭으로 판단
3. **하이브리드**: Dice로 후보군 축소 → Edit Distance로 최종 확인
4. **향후**: Sentence Embedding 기반으로 발전 가능

In [ ]:
# 개선된 매칭: 정규화 + Bigram Dice
def improved_matching(query, candidates, threshold=0.5):
    """정규화 전처리 + Bigram Dice 매칭"""
    query_norm = normalize_store_name(query)
    
    scores = []
    for candidate in candidates:
        candidate_norm = normalize_store_name(candidate)
        score = bigram_dice(query_norm, candidate_norm)
        scores.append((candidate, score))
    
    scores.sort(key=lambda x: -x[1])
    best_match, best_score = scores[0]
    
    if best_score >= threshold:
        return best_match, best_score
    else:
        return None, best_score

# 개선된 매칭 실험
print("=== 개선된 매칭 (정규화 + Bigram Dice) ===")
print(f"{'쿼리':<20} {'정답':<12} {'예측':<12} {'점수':>6} {'결과'}")
print("-" * 60)

improved_correct = 0
for query, answer, category in match_tests:
    match, score = improved_matching(query, store_names)
    is_correct = match == answer
    if is_correct:
        improved_correct += 1
    status = 'O' if is_correct else 'X'
    match_str = match if match else '(매칭 실패)'
    print(f"{query:<20} {answer:<12} {match_str:<12} {score:>6.3f} {status}")

print(f"\n정확도: {improved_correct}/{len(match_tests)} ({improved_correct/len(match_tests)*100:.1f}%)")
print(f"\n→ 정규화 전처리를 추가하면 지점명 제거 문제가 개선됨")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Overlap Coefficient 구현

Overlap Coefficient는 더 짧은 문자열의 관점에서 유사도를 측정합니다.

$$\text{Overlap}(A, B) = \frac{|A \cap B|}{\min(|A|, |B|)}$$

"이디야커피" vs "이디야" 같은 **부분 매칭**에 강할 수 있습니다.

In [ ]:
# TODO: Overlap Coefficient를 bigram 기반으로 구현하세요
#
# 1. overlap_coefficient(s1, s2) 함수 작성
# 2. 위의 test_cases에 대해 결과를 출력하세요
# 3. Dice coefficient와 비교하여 어떤 케이스에서 차이가 나는지 분석하세요
#
# 특히 "이디야커피" vs "이디야" 같은 부분 매칭 케이스에서의 차이에 주목하세요.


### 연습 2: 하이브리드 매칭 시스템 구현

여러 유사도 방식을 결합한 하이브리드 매칭 시스템을 구현하세요.

In [ ]:
# TODO: 하이브리드 매칭 시스템
#
# 아이디어: 여러 유사도의 가중 합을 사용
#   score = w1 * dice + w2 * edit_sim + w3 * jaccard
#
# 1. hybrid_match(query, candidates, weights) 함수를 구현하세요
#    - weights: {'dice': 0.5, 'edit': 0.3, 'jaccard': 0.2} 등
#
# 2. match_tests에 대해 여러 가중치 조합을 실험하세요
#    - 어떤 가중치 조합이 가장 높은 정확도를 보이는지?
#
# 3. 정규화 전처리와 결합했을 때의 결과도 확인하세요


---
## 핵심 정리

| 개념 | 설명 | 적합한 상황 |
|------|------|-------------|
| Edit Distance | 최소 편집 횟수 (DP) | 오타 교정, 짧은 문자열 |
| Jaccard | N-gram 집합의 교집합/합집합 | 일반적 문자열 비교 |
| Dice Coefficient | Jaccard보다 교집합에 더 가중치 | **매장명 매칭 (ai-ipsonum)** |
| Cosine (TF-IDF) | 문서 벡터 간 각도 | 긴 문서 비교 |
| Cosine (Embedding) | 밀집 벡터 간 각도 | 의미적 유사도 |
| 전처리 + 유사도 | 정규화 후 유사도 적용 | **실무 최적 조합** |

**다음 노트북**: [04-seq2seq-attention.ipynb](04-seq2seq-attention.ipynb) - Seq2Seq + Attention